<a href="https://colab.research.google.com/github/FeLocci/senacai/blob/main/C%C3%B3pia_de_analise_de_som_desafio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import kagglehub
import os
import numpy as np
import librosa
import librosa.display
import tensorflow as tf
from tensorflow.keras import layers, models, applications
from sklearn.model_selection import train_test_split
import os
import pandas as pd # Import pandas for reading metadata
import random

# 1. DOWNLOAD
path = kagglehub.dataset_download("soumendraprasad/musical-instruments-sound-dataset")
# The actual audio files for training are in a subfolder structure, typically:
# <download_path>/Train_submission/Train_submission/
# The metadata (labels) are in:
# <download_path>/Train_submission/Metadata_Train.csv

# Print path information for debugging
print("Dataset downloaded to:", path)
print("Content of download directory:", os.listdir(path))
print("Content of Train_submission directory:", os.listdir(os.path.join(path, "Train_submission/Train_submission")))
print(os.listdir(os.path.join(path, "Train_submission")))


100%|██████████| 5.40G/5.40G [02:19<00:00, 41.5MB/s]

Extracting files...


Dataset downloaded to: /root/.cache/kagglehub/datasets/soumendraprasad/musical-instruments-sound-dataset/versions/3
Content of download directory: ['Test_submission', 'Train_submission', 'Metadata_Train.csv', 'Metadata_Test.csv']
Content of Train_submission directory: ['LP_A_muted5.wav', '066166_qui-c39est-qu39est-tombe-loop-t85wav-39366.wav', 'G53-69605-1111-305.wav', 'AR_D_fret_0-20.wav', 'ROOM_room5_MUS_pachelbel_DEV_lg.wav', 'slow_classical_5_70BPM.wav', 'AR_Lick2_KN.wav', '1_ray-chen_bwv1004_mov5.wav', 'Vn-ord-A6-mf-1c.wav', 'VIOLIN_sound (323).wav', 'violin_sound (41).wav', 'ROOM_room8_MUS_chords_DEV_ipad.wav', 'FS_Lick1_MN.wav', 'LP_Lick11_FN.wav', '8_emil-telmanyi_bwv1001.wav', 'Violin_Sound (296).wav', 'WaveDrum02_45SD (9).wav', 'joyful-messy-piano-116715.wav', 'WaveDrum02_39KD (21).wav', 'G53-47202-1111-172.wav', 'TechnoDrum01_03SDtrain.wav', 'violin_sound (43).wav', 'WaveDrum02_39KD (20).wav', 'LP_Lick6_FHBV.wav', 'slow_ska_2_150BPM.wav', 'pathetique_poly.wav', 'LP_Lick2_FN.

In [5]:
# Function for transforming an audio file into a image

def audio_to_spectrogram(audio_path):
    # Carrega apenas 3 segundos para economizar muita RAM
    #y, sr = librosa.load(audio_path, duration=3)
    y = librosa.load(audio_path)

    # Handle cases where audio is too short or silent
    if len(y) == 0 or np.all(y == 0):
        # Return an array of zeros with the expected shape (height, width, channels)
        # The .numpy() conversion will happen outside for consistency with other data
        return tf.zeros(IMG_SIZE + (3,), dtype=tf.float32)

    S = librosa.feature.melspectrogram(y=y, n_mels=128)
    # Convert to dB, adding a small constant to prevent log(0) and clipping values below top_db
    S_dB = librosa.power_to_db(S + 1e-8, ref=np.max, top_db=80.0)

    # Normalize S_dB to a 0-1 range
    min_val = np.min(S_dB)
    max_val = np.max(S_dB)

    if max_val == min_val: # If the spectrogram is flat (e.g., all silence after dB conversion)
        normalized_S_dB = np.zeros_like(S_dB)
    else:
        normalized_S_dB = (S_dB - min_val) / (max_val - min_val)

    # Redimensiona e converte para RGB (3 canais) para o MobileNet
    img = tf.image.resize(np.expand_dims(normalized_S_dB, -1), IMG_SIZE)
    img = tf.image.grayscale_to_rgb(img)
    return img

In [6]:
# Configurações de Memória
IMG_SIZE = (224, 224)
CLASSES = ['Sound_Guitar', 'Sound_Drum', 'Sound_Violin', 'Sound_Piano'] # Corrected class labels to match metadata
MAX_FILES_PER_CLASS = 60  # Reduzimos aqui para garantir que a RAM aguente


# Define the base paths for audio files and metadata
# 'path' variable is expected to be available from the previous cell's execution.
# Based on typical Kaggle dataset structure and previous outputs:
audio_files_base_path = os.path.join(path, "Train_submission", "Train_submission")
metadata_file_path = os.path.join(path, "Metadata_Train.csv") # Corrected path for metadata file

x_data, y_data = [], []
label_to_int = {label: i for i, label in enumerate(CLASSES)}

# Load metadata to get file names and their corresponding labels
if not os.path.exists(metadata_file_path):
    print(f"Error: Metadata file not found at '{metadata_file_path}'. Cannot load data.")
else:
    metadata_df = pd.read_csv(metadata_file_path)
    print(f"Metadata loaded from: {metadata_file_path}")

    # Iterate through each class to load a limited number of files
    for label_str in CLASSES:
        class_df = metadata_df[metadata_df['Class'] == label_str].head(MAX_FILES_PER_CLASS)
        print(f"Lendo {len(class_df)} arquivos para a classe '{label_str}'...")

        for index, row in class_df.iterrows():
            file_name = row['FileName']
            full_audio_path = os.path.join(audio_files_base_path, file_name)

            if not os.path.exists(full_audio_path):
                print(f"Warning: Audio file not found at '{full_audio_path}'. Skipping.")
                continue

            try:
                spec = audio_to_spectrogram(full_audio_path)
                x_data.append(spec.numpy()) # Converte para numpy para economizar memória de GPU
                y_data.append(label_to_int[label_str])
            except Exception as e:
                print(f"Error processing {full_audio_path}: {e}. Skipping.")

X = np.array(x_data)
y = np.array(y_data)

if len(X) == 0:
    print("Error: No audio data was successfully loaded. Please check dataset paths and contents.")
    # Prevent ValueError from train_test_split with empty data
    X_train, X_test, y_train, y_test = np.array([]), np.array([]), np.array([]), np.array([])
else:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y) # Added stratify


print(f"\nSucesso! {len(X)} imagens processadas.")
print(f"Formato do array X_train: {X_train.shape}")
print(f"Formato do array y_train: {y_train.shape}")
print(f"Formato do array X_test: {X_test.shape}")
print(f"Formato do array y_test: {y_test.shape}")


Metadata loaded from: /root/.cache/kagglehub/datasets/soumendraprasad/musical-instruments-sound-dataset/versions/3/Metadata_Train.csv
Lendo 60 arquivos para a classe 'Sound_Guitar'...
Error processing /root/.cache/kagglehub/datasets/soumendraprasad/musical-instruments-sound-dataset/versions/3/Train_submission/Train_submission/1-E1-Major 00.wav: Audio data must be of type numpy.ndarray. Skipping.
Error processing /root/.cache/kagglehub/datasets/soumendraprasad/musical-instruments-sound-dataset/versions/3/Train_submission/Train_submission/1-E1-Major 01.wav: Audio data must be of type numpy.ndarray. Skipping.
Error processing /root/.cache/kagglehub/datasets/soumendraprasad/musical-instruments-sound-dataset/versions/3/Train_submission/Train_submission/1-E1-Major 02.wav: Audio data must be of type numpy.ndarray. Skipping.
Error processing /root/.cache/kagglehub/datasets/soumendraprasad/musical-instruments-sound-dataset/versions/3/Train_submission/Train_submission/1-E1-Major 03.wav: Audio da

In [4]:
from re import VERBOSE


# Criando o modelo baseado em MobileNetV2 - nesse momento estamos usando o conceito MobileNetV2
base_model = applications.MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(len(CLASSES), activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

print("Iniciando treinamento final...")
history = model.fit(X_train, y_train, epochs=10, validation_data=(X_test, y_test), batch_size=16, verbose=1)

model.summary()
print(history.history['loss'])

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Iniciando treinamento final...
Epoch 1/10


ValueError: Exception encountered when calling Sequential.call().

[1mInvalid input shape for input Tensor("data:0", shape=(16,), dtype=float32) with name 'keras_tensor_154' and path ''. Expected shape (None, 224, 224, 3), but input has incompatible shape (16,)[0m

Arguments received by Sequential.call():
  • inputs=tf.Tensor(shape=(16,), dtype=float32)
  • training=True
  • mask=None
  • kwargs=<class 'inspect._empty'>

In [ ]:
def select_file():# Filter list to include only files
  # Uses test submission files
  dir_test = os.path.join(path, "Test_submission", "Test_submission")
  metdata_test_file=os.path.join(path, "Metadata_Test.csv")


  # Select random file
  files = [f for f in os.listdir(dir_test) if os.path.isfile(os.path.join(dir_test, f))]
  if files:
      random_file = random.choice(files)
      print(f"Selected: {random_file}\n")
      # Filter rows where a specific column matches a regex pattern
      df=pd.read_csv(metdata_test_file)
      matched_rows = df[df['FileName'].str.contains(random_file, na=False, regex=True)]
      # Print the resulting rows
      print(matched_rows,"\n")
      random_file = os.path.join(dir_test, random_file)
  else:
      print("No files found in directory.")


  if not os.path.isfile(random_file):
      print(f"Arquivo não encontrado: {random_file}")
      os.sys.exit(1)
  return random_file, matched_rows



In [ ]:
import pandas as pd

all_results = [] # Lista para armazenar os resultados de cada iteração

for i in range(10): # Loop 10 vezes
  print(f"\n--- Iteração {i+1}/10 ---")
  random_file, matched_rows = select_file() # Chamar a função para selecionar um novo arquivo e metadados
  print(f"Processando o arquivo de áudio: {os.path.basename(random_file)}")

  try:
      # `audio_to_spectrogram` já converte para o tamanho e 3 canais
      spectrogram_image = audio_to_spectrogram(random_file)
      # Adiciona a dimensão do batch para o modelo
      input_spectrogram = np.expand_dims(spectrogram_image, axis=0)

      # 2. Make a prediction using the trained model
      predictions = model.predict(input_spectrogram)
      predicted_class_index = np.argmax(predictions)
      predicted_class_label = CLASSES[predicted_class_index]

      # 3. Get the true class label from matched_rows
      if not matched_rows.empty and 'Class' in matched_rows.columns:
          true_class_label = matched_rows['Class'].iloc[0]
      else:
          true_class_label = "Unknown (metadata not found)"
          print("Warning: Could not find true class label for the selected file.")

      # 4. Compare prediction with true class
      is_correct = (predicted_class_label == true_class_label)
      result = "Correct" if is_correct else "Incorrect"

      # 5. Store results in a DataFrame (for this iteration)
      iteration_results_df = pd.DataFrame({
          'File Name': [os.path.basename(random_file)],
          'True Class': [true_class_label],
          'Predicted Class': [predicted_class_label],
          'Prediction Result': [result]
      })
      all_results.append(iteration_results_df)

      # Print immediate feedback for each iteration
      if not is_correct:
          print(f"Oh não! A predição para '{os.path.basename(random_file)}' está INCORRETA.")
          print(f"O modelo previu '{predicted_class_label}' mas a classe verdadeira é '{true_class_label}'.")
      else:
          print(f"Excelente! A predição para '{os.path.basename(random_file)}' está CORRETA.")

  except Exception as e:
      print(f"Erro ao processar e predizer o arquivo {os.path.basename(random_file)}: {e}")

# Concatenar todos os resultados em um único DataFrame no final
if all_results:
    final_results_df = pd.concat(all_results, ignore_index=True)
    print("\n--- Resultados Finais de Todas as Predições ---")
    display(final_results_df)
else:
    print("Nenhuma predição foi realizada com sucesso.")
